# Evaluate Features: IC Scoring, Purged K-Fold CV, and Regime Backtesting

This notebook evaluates feature predictive power using **Information Coefficient (IC) analysis**, demonstrates **leakage-free cross-validation** with Purged K-Fold, and compares backtest results **with and without regime filtering**.

IC (Information Coefficient) measures the rank correlation between a feature value and future returns. A positive IC means the feature tends to predict positive forward returns; a negative IC means it predicts negative returns. IC values above |0.05| at multiple horizons are considered useful in practice.

---

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Setup](#setup)
3. [Parameters](#parameters)
4. [DB Connection and Feature Load](#db-connection)
5. [Part 1: IC Evaluation](#ic-evaluation)
   - [What is the Information Coefficient?](#ic-intro)
   - [Single Feature IC](#single-ic)
   - [IC Decay Chart](#ic-decay)
   - [Rolling IC Over Time](#rolling-ic)
   - [Batch IC: Multiple Features](#batch-ic)
6. [Part 2: Purged K-Fold Cross-Validation](#purged-kfold)
   - [Why Purged K-Fold?](#kfold-intro)
   - [Build Splitter and Visualize](#kfold-splits)
   - [Fold Statistics](#fold-stats)
7. [Part 3: Regime A/B Backtest Comparison](#regime-backtest)
   - [RSI Signal Generation](#rsi-signals)
   - [Regime-Filtered Entries](#regime-filter)
   - [Run Both Backtests](#run-backtests)
   - [Comparison Table](#comparison-table)
   - [Regime-Conditional IC](#regime-ic)
8. [Summary and Next Steps](#summary)

<a id='prerequisites'></a>
## Prerequisites

**Required database tables:**
- `cmc_features` — bar-level feature store (112 columns including `rsi_14`, `close`, returns, vol, TA features)
- `cmc_regimes` — regime labels (l0, l1, l2) per bar
- `cmc_ic_results` — IC results table (optional; used if saving IC results to DB)
- `cmc_ema_multi_tf_u` — EMA values (queried by signal generators; NOT in cmc_features)
- `asset_data_coverage` — data coverage summary

**Required CLI (run before opening notebook):**
```bash
# Refresh feature store for asset 1, 1D timeframe
python -m ta_lab2.scripts.features.run_all_feature_refreshes --all --tf 1D
```

**Note on EMA columns:** EMA values (ema_9, ema_21, etc.) are **not** stored in `cmc_features` — they live in `cmc_ema_multi_tf_u`. Signal generators query `cmc_ema_multi_tf_u` directly via LEFT JOINs. This notebook uses `rsi_14` from `cmc_features` for regime A/B backtest.

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd().parent
if str(_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_ROOT / "src"))

import helpers
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import ta_lab2
print(f"ta_lab2 loaded from: {ta_lab2.__file__}")

<a id='parameters'></a>
## Parameters

Change `ASSET_ID`, `TF`, `START_DATE`, or `END_DATE` to analyse a different asset. Re-run all cells after changing parameters.

- `HORIZONS` — forward return horizons in bars used for IC evaluation
- `RETURN_TYPES` — arithmetic and/or log returns
- `ROLLING_WINDOW` — window size for rolling IC (63 bars ≈ 1 quarter for daily data)
- `N_SPLITS` — number of folds for purged K-fold
- `EMBARGO_FRAC` — fraction of sample to embargo after each test fold

In [ ]:
ASSET_ID = 1
TF = "1D"
START_DATE = "2021-01-01"
END_DATE = "2025-12-31"
HORIZONS = [1, 2, 3, 5, 10, 20, 60]
RETURN_TYPES = ["arith", "log"]
ROLLING_WINDOW = 63
N_SPLITS = 5
EMBARGO_FRAC = 0.01

<a id='db-connection'></a>
## DB Connection and Feature Load

We create a single NullPool engine (no connection pooling — appropriate for notebooks) and load features from `cmc_features`. Timestamps are always UTC-aware after loading.

In [ ]:
engine = helpers.get_engine()

# Validate data coverage
check = helpers.validate_asset_data(engine, ASSET_ID, TF, min_days=365)
print(f"Asset {ASSET_ID} ({TF}): {check['message']}")
if not check["valid"]:
    raise RuntimeError(f"Insufficient data: {check['message']}")

# Load feature DataFrame
features_df = helpers.load_features(engine, ASSET_ID, TF, START_DATE, END_DATE)
if features_df.empty:
    raise RuntimeError(
        f"No features found for asset {ASSET_ID} / {TF} in [{START_DATE}, {END_DATE}]. "
        "Run: python -m ta_lab2.scripts.features.run_all_feature_refreshes --all --tf 1D"
    )

# Ensure timestamps are tz-aware UTC (Windows pitfall: pd.read_sql may return object dtype)
if not hasattr(features_df.index, "tz") or features_df.index.tz is None:
    features_df.index = pd.to_datetime(features_df.index, utc=True)

print(f"Loaded {len(features_df)} feature rows from {features_df.index.min()} to {features_df.index.max()}")
print(f"Columns: {len(features_df.columns)} total. Feature sample: {list(features_df.columns[:8])}")

# IC evaluation boundaries (tz-aware Timestamps)
train_start = pd.to_datetime(START_DATE, utc=True)
train_end = pd.to_datetime(END_DATE, utc=True)

# Separate close series for IC computation
close = features_df["close"].dropna()
print(f"\nClose series: {len(close)} bars, range [{close.min():.2f}, {close.max():.2f}]")

features_df.head()

<a id='ic-evaluation'></a>
---
# Part 1: IC Evaluation

<a id='ic-intro'></a>
## What is the Information Coefficient?

The **Information Coefficient (IC)** is the **Spearman rank correlation** between a feature value at time `t` and the forward return at time `t+h` (where `h` is the horizon).

Using rank correlation (Spearman) instead of linear correlation (Pearson) makes IC robust to outliers and non-linear monotone relationships — exactly what we want for financial features.

**Interpretation guide:**

| IC Range | Interpretation |
|----------|----------------|
| \|IC\| < 0.02 | Negligible — noise level |
| \|IC\| 0.02–0.05 | Weak but potentially useful with ensemble |
| \|IC\| 0.05–0.10 | Useful — most deployed quant signals live here |
| \|IC\| > 0.10 | Strong — rare, warrants double-checking for leakage |

**Key design decisions in ta_lab2's IC library:**

1. **`train_start` / `train_end` are required** — IC without a bounded evaluation window risks future-information leakage (e.g., forward returns near the end of the series use prices after the evaluation window).

2. **Boundary masking** — Bars where `bar_ts + horizon_days > train_end` have their forward returns nulled out. Without this, the last `h` bars in the window use prices from *after* `train_end`.

3. **IC-IR (Information Ratio)** — `IC-IR = mean(rolling_IC) / std(rolling_IC)`. High IC-IR means the feature produces consistent IC over time, not just one lucky period.

4. **IC Decay** — A feature with high IC at h=1 but decaying IC by h=20 is a short-term signal. One that maintains IC across horizons is a trend-following signal.

<a id='single-ic'></a>
## Single Feature IC Evaluation

We start with `rsi_14` — a standard momentum oscillator available in `cmc_features`. The IC computation evaluates Spearman correlation between `rsi_14` at each bar and the forward arithmetic/log return `h` bars ahead.

In [ ]:
from ta_lab2.analysis.ic import compute_ic, compute_rolling_ic, compute_forward_returns

FEATURE_COL = "rsi_14"

# Validate feature column exists
if FEATURE_COL not in features_df.columns:
    available = [c for c in features_df.columns if "rsi" in c.lower()]
    raise ValueError(f"Column '{FEATURE_COL}' not found. RSI-like columns: {available}")

feature_series = features_df[FEATURE_COL].dropna()
print(f"Feature: {FEATURE_COL}")
print(f"  Bars: {len(feature_series)}")
print(f"  Range: [{feature_series.min():.2f}, {feature_series.max():.2f}]")
print(f"  Mean: {feature_series.mean():.2f}  Std: {feature_series.std():.2f}")

# compute_ic requires tz-aware train_start / train_end
ic_df = compute_ic(
    feature_series,
    close,
    train_start,
    train_end,
    horizons=HORIZONS,
    return_types=RETURN_TYPES,
    rolling_window=ROLLING_WINDOW,
    tf_days_nominal=1,
)

print(f"\nIC results: {len(ic_df)} rows ({len(HORIZONS)} horizons x {len(RETURN_TYPES)} return types)")
ic_df

**Reading the IC table:**

- `ic` — Spearman rank correlation. Positive values mean the feature predicts positive returns.
- `ic_t_stat` / `ic_p_value` — Statistical significance. `ic_p_value < 0.05` means the IC is unlikely to be zero by chance.
- `ic_ir` — IC Information Ratio. Higher is better; `ic_ir > 0.5` is considered useful.
- `ic_ir_t_stat` — t-statistic for IC-IR ≠ 0. Measures whether rolling IC is reliably positive.
- `turnover` — Rank autocorrelation proxy. High turnover (> 0.5) means signal ranks change frequently, increasing transaction costs.
- `n_obs` — Number of valid (non-NaN) observations used.

<a id='ic-decay'></a>
## IC Decay Chart

The IC decay chart shows how predictive power **decays as the forward horizon increases**. Blue bars are statistically significant (`p < 0.05`); gray bars are not.

**What to look for:**
- A **mean-reversion signal** (like RSI) typically shows negative IC at short horizons (negative RSI → near-term bounce) and IC decaying toward zero at longer horizons.
- A **trend-following signal** shows positive IC that persists across horizons.
- **Inverted V shape** — IC peaks at some intermediate horizon, then decays. Common for momentum signals.

In [ ]:
from ta_lab2.analysis.ic import plot_ic_decay

# Plot IC decay for arithmetic returns
fig_decay_arith = plot_ic_decay(
    ic_df,
    feature=FEATURE_COL,
    return_type="arith",
    sig_threshold=0.05,
)
fig_decay_arith.show()

# Plot IC decay for log returns
fig_decay_log = plot_ic_decay(
    ic_df,
    feature=FEATURE_COL,
    return_type="log",
    sig_threshold=0.05,
)
fig_decay_log.show()

**Reading the IC decay chart:**

- **Blue bars** — statistically significant at p < 0.05. These horizons have detectable predictive signal.
- **Gray bars** — not significant; IC could plausibly be zero.
- **Negative IC at short horizons + positive at long horizons** — classic mean-reversion pattern for RSI. High RSI → short-term pullback (negative IC) but longer-term continuation.
- **IC magnitude shrinks toward zero** — expected. Market returns become harder to predict further out.

Arithmetic and log return IC values are almost identical for small returns but can diverge for large moves (crypto has large daily moves, so log returns are often more appropriate).

<a id='rolling-ic'></a>
## Rolling IC Over Time

A single point-in-time IC might be driven by one unusual period (e.g., a crash). **Rolling IC** shows whether the feature's predictive power is stable over time or concentrated in specific regimes.

We compute rolling Spearman IC using the rank-then-correlate pattern (`rolling().rank()` then `rolling().corr()`), which is ~30x faster than per-window `spearmanr()` calls.

In [ ]:
from ta_lab2.analysis.ic import plot_rolling_ic

ROLLING_HORIZON = 5  # h=5 is a common medium-term horizon for daily signals

# Compute forward returns for the chosen horizon
fwd_ret = compute_forward_returns(close, horizon=ROLLING_HORIZON, log=False)

# Slice to train window
train_mask = (feature_series.index >= train_start) & (feature_series.index <= train_end)
feat_train = feature_series[train_mask]
fwd_train = fwd_ret.reindex(feat_train.index).copy()

# Boundary masking
horizon_delta = pd.Timedelta(days=ROLLING_HORIZON)
boundary_mask = (feat_train.index + horizon_delta) > train_end
fwd_train.iloc[boundary_mask] = np.nan

rolling_ic_series, ic_ir, ic_ir_tstat = compute_rolling_ic(
    feat_train, fwd_train, window=ROLLING_WINDOW
)

print(f"Rolling IC (horizon={ROLLING_HORIZON}, window={ROLLING_WINDOW}):")
print(f"  IC-IR:        {ic_ir:.4f}" if not np.isnan(ic_ir) else "  IC-IR:        NaN (insufficient data)")
print(f"  IC-IR t-stat: {ic_ir_tstat:.4f}" if not np.isnan(ic_ir_tstat) else "  IC-IR t-stat: NaN")
print(f"  Valid rolling IC observations: {rolling_ic_series.notna().sum()}")

# Plot rolling IC
fig_rolling = plot_rolling_ic(
    rolling_ic_series,
    feature=FEATURE_COL,
    horizon=ROLLING_HORIZON,
    return_type="arith",
)
fig_rolling.show()

**IC-IR (Information Ratio) interpretation:**

- **IC-IR > 0.5** — The feature's rolling IC is reliably above zero. Good enough for ensemble use.
- **IC-IR 0.3–0.5** — Marginally useful; consider combining with other features.
- **IC-IR < 0.3** — Unstable. IC might be positive overall but driven by regime-specific periods.

When rolling IC oscillates between positive and negative, it indicates the feature's signal **reverses across regimes** — a strong motivation for regime-conditional analysis (Part 3).

The IC-IR t-statistic tests whether the mean rolling IC is significantly different from zero: `t = mean(IC) * sqrt(n) / std(IC)`.

<a id='batch-ic'></a>
## Batch IC: Multiple Features

`batch_compute_ic()` evaluates IC for multiple feature columns in one call. It auto-discovers numeric columns (excluding `close`) or accepts an explicit `feature_cols` list.

We evaluate a curated subset of features to keep runtime manageable in the notebook.

In [ ]:
from ta_lab2.analysis.ic import batch_compute_ic

# Curated feature subset for batch IC — covers momentum, volatility, and TA categories
_candidate_cols = [
    "rsi_14",
    "ret_arith",
    "vol_30d",
    "atr_14",
    "bb_width",
    "macd_hist",
    "ret_arith_zscore_30",
]

# Keep only columns that actually exist in features_df
BATCH_COLS = [c for c in _candidate_cols if c in features_df.columns]

if not BATCH_COLS:
    # Fallback: take first 5 numeric columns that are not 'close'
    BATCH_COLS = [
        c for c in features_df.select_dtypes(include=[np.number]).columns
        if c not in ("close", "id")
    ][:5]

print(f"Running batch IC on {len(BATCH_COLS)} features: {BATCH_COLS}")
print("(horizons=all, return_type=arith, this may take 10-30 seconds...)")

batch_df = batch_compute_ic(
    features_df,
    close,
    train_start,
    train_end,
    feature_cols=BATCH_COLS,
    horizons=HORIZONS,
    return_types=["arith"],  # arith only for speed
    rolling_window=ROLLING_WINDOW,
    tf_days_nominal=1,
)

print(f"\nBatch IC complete: {len(batch_df)} rows ({len(BATCH_COLS)} features x {len(HORIZONS)} horizons)")
batch_df.head()

In [ ]:
# Pivot to feature x horizon table for easy comparison
pivot_ic = batch_df.pivot_table(
    index="feature",
    columns="horizon",
    values="ic",
    aggfunc="mean",
).round(4)
pivot_ic.columns = [f"h={c}" for c in pivot_ic.columns]

# Add IC-IR column (mean across horizons per feature)
pivot_ic_ir = batch_df.groupby("feature")["ic_ir"].mean().round(4).rename("mean_IC-IR")
pivot_combined = pivot_ic.join(pivot_ic_ir)

print("IC Decay Table — Spearman IC by feature and horizon (arith returns)")
print("Blue = significant (p<0.05), Gray = not significant")
print()

# Build significance pivot for coloring
pivot_pval = batch_df.pivot_table(
    index="feature",
    columns="horizon",
    values="ic_p_value",
    aggfunc="mean",
)

# Apply styling: RdYlGn gradient on IC values, bold where significant
def highlight_sig(val):
    """Highlight positive IC green, negative IC red (significance via intensity)."""
    if pd.isna(val):
        return ""
    if val > 0.05:
        return "color: royalblue; font-weight: bold"
    elif val < -0.05:
        return "color: crimson; font-weight: bold"
    return "color: gray"

ic_cols = [c for c in pivot_combined.columns if c.startswith("h=")]
styled = (
    pivot_combined.style
    .background_gradient(subset=ic_cols, cmap="RdYlGn", vmin=-0.08, vmax=0.08)
    .format("{:.4f}", subset=ic_cols)
    .format("{:.4f}", subset=["mean_IC-IR"])
)
display(styled)  # noqa: F821 — Jupyter built-in

**Reading the batch IC table:**

- Each row is a feature; each column is a forward horizon.
- **Red gradient** — negative IC at that horizon. Feature predicts negative returns.
- **Green gradient** — positive IC at that horizon. Feature predicts positive returns.
- **`mean_IC-IR`** — average IC Information Ratio across all horizons. Higher is more consistent.

For RSI (mean-reversion): expect negative IC at short horizons (oversold → bounce prediction is backwards for raw RSI — remember, high RSI means overbought, so IC should be negative if you're predicting the pullback from the long side).

For momentum features (`ret_arith`): expect positive IC at short-to-medium horizons (recent returns predict continuation).

<a id='purged-kfold'></a>
---
# Part 2: Purged K-Fold Cross-Validation

<a id='kfold-intro'></a>
## Why Purged K-Fold?

Standard K-Fold CV **leaks future information** in financial time series because:

1. **Overlapping labels**: Multi-bar holding period returns overlap. A label starting at `t` with a 5-day horizon uses prices at `t+1, ..., t+5`. If `t+3` is in the training set, you're training on data that overlaps with your test label.

2. **Serial autocorrelation**: Financial features are autocorrelated. A training observation at `t+1` (just after the test fold ends) contains information from the test fold period.

**Purged K-Fold** solves this with two steps:

- **Purge**: Remove training observations whose *label-end timestamp* (`t1`) extends into the test fold window.
- **Embargo**: Remove training observations in a gap window `[test_end, test_end + embargo_size)` to prevent serial autocorrelation leakage.

```
         |---TRAIN---| PURGE |---TEST---| EMBARGO |---TRAIN---|
         |           |XXXXX |          |  XXXXX  |           |
```

The `t1_series` represents label-end timestamps. For a 1-bar holding period, `t1[i] = index[i+1]`. For a 5-bar holding period, `t1[i] = index[i+5]`.

In [ ]:
from ta_lab2.backtests.cv import PurgedKFoldSplitter

# Build t1_series for a 5-bar holding period (label-end timestamps)
# t1[i] = the timestamp 5 bars after observation i
LABEL_HORIZON = 5  # bars

feat_index = features_df.index
n = len(feat_index)

# For each bar at position i, the label ends at position i + LABEL_HORIZON
t1_values = []
for i in range(n):
    end_pos = min(i + LABEL_HORIZON, n - 1)
    t1_values.append(feat_index[end_pos])

t1_series = pd.Series(t1_values, index=feat_index)
print(f"t1_series: {len(t1_series)} entries")
print(f"  Index range: {t1_series.index.min()} to {t1_series.index.max()}")
print(f"  Values range: {t1_series.min()} to {t1_series.max()}")
print(f"  Label horizon: {LABEL_HORIZON} bars (t1 leads index by up to {LABEL_HORIZON} bars)")

# Build PurgedKFoldSplitter
splitter = PurgedKFoldSplitter(
    n_splits=N_SPLITS,
    t1_series=t1_series,
    embargo_frac=EMBARGO_FRAC,
)
print(f"\nPurgedKFoldSplitter created: {N_SPLITS} folds, embargo_frac={EMBARGO_FRAC}")

# Generate splits
X_dummy = np.zeros((n, 1))  # splitter uses len(X) only
splits = list(splitter.split(X_dummy))
print(f"Generated {len(splits)} fold splits")

<a id='kfold-splits'></a>
## Visualize Splits

Each horizontal bar shows one fold. Colors indicate:
- **Blue** — training observations
- **Orange** — test fold
- **Red** — purged observations (label bleeds into test)
- **Gray** — embargo zone (serial autocorrelation buffer)

In [ ]:
from ta_lab2.backtests.cv import fold_boundaries

# Classify each observation for each fold
embargo_size = max(1, int(EMBARGO_FRAC * n)) if EMBARGO_FRAC > 0 else 0
fold_bounds = fold_boundaries(n, N_SPLITS)

fig_splits, ax = plt.subplots(figsize=(14, 5))

colors = {"train": "#4C72B0", "test": "#DD8452", "purge": "#C44E52", "embargo": "#BBBBBB"}

for fold_idx, (train_idx, test_idx) in enumerate(splits):
    fold_start, fold_end = fold_bounds[fold_idx]
    test_start_ts = t1_series.index[fold_start]
    test_end_pos = fold_end - 1
    embargo_start = test_end_pos + 1
    embargo_end = min(test_end_pos + embargo_size, n - 1)

    # Build classification per bar
    bar_class = np.full(n, "train")  # default: training
    bar_class[fold_start:fold_end] = "test"  # test fold

    # Purged: not in train_idx (complement - train_idx)
    complement = np.concatenate([np.arange(0, fold_start), np.arange(fold_end, n)])
    train_set = set(train_idx.tolist())
    for pos in complement:
        if pos not in train_set:
            if embargo_start <= pos <= embargo_end:
                bar_class[pos] = "embargo"
            else:
                bar_class[pos] = "purge"

    # Draw horizontal bars
    y = fold_idx
    start = 0
    while start < n:
        current = bar_class[start]
        end = start + 1
        while end < n and bar_class[end] == current:
            end += 1
        ax.barh(
            y,
            end - start,
            left=start,
            height=0.7,
            color=colors[current],
            align="center",
        )
        start = end

# Legend
legend_patches = [
    mpatches.Patch(color=colors["train"], label="Training"),
    mpatches.Patch(color=colors["test"], label="Test"),
    mpatches.Patch(color=colors["purge"], label="Purged (label overlap)"),
    mpatches.Patch(color=colors["embargo"], label=f"Embargo ({embargo_size} bars)"),
]
ax.legend(handles=legend_patches, loc="upper right", fontsize=9)

ax.set_yticks(range(N_SPLITS))
ax.set_yticklabels([f"Fold {i+1}" for i in range(N_SPLITS)])
ax.set_xlabel("Bar index")
ax.set_title(
    f"Purged K-Fold Splits — Asset {ASSET_ID} ({TF}), {N_SPLITS} folds, "
    f"horizon={LABEL_HORIZON}, embargo_frac={EMBARGO_FRAC}"
)
plt.tight_layout()
plt.show()

print(f"\nSplit sizes (train | test | purged+embargo):")
for i, (train_idx, test_idx) in enumerate(splits):
    n_train = len(train_idx)
    n_test = len(test_idx)
    n_other = n - n_train - n_test
    print(f"  Fold {i+1}: train={n_train}, test={n_test}, purged+embargo={n_other}")

**Reading the splits chart:**

- Each row is a fold where that fold's observations are used as the **test set**.
- **Purged (red)** zones are the training observations immediately before the test fold whose labels overlap the test window. These are removed from training to prevent leakage.
- **Embargo (gray)** zones are the training observations immediately after the test fold. These are removed to prevent serial autocorrelation leakage.
- The effective training set is smaller than `(n - n_test)` due to purge and embargo — this is the cost of leakage-free evaluation.

Standard K-Fold would include both red and gray observations in the training set, creating look-ahead bias.

<a id='fold-stats'></a>
## Fold Statistics

Detailed statistics for each fold — training size, test size, and date ranges.

In [ ]:
fold_stats = []

for fold_idx, (train_idx, test_idx) in enumerate(splits):
    test_start_ts = feat_index[test_idx[0]]
    test_end_ts = feat_index[test_idx[-1]]
    train_start_ts = feat_index[train_idx[0]] if len(train_idx) > 0 else None
    train_end_ts = feat_index[train_idx[-1]] if len(train_idx) > 0 else None

    n_purge_embargo = n - len(train_idx) - len(test_idx)
    pct_train = 100 * len(train_idx) / n
    pct_test = 100 * len(test_idx) / n

    fold_stats.append(
        {
            "fold": fold_idx + 1,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_purge_embargo": n_purge_embargo,
            "pct_train": round(pct_train, 1),
            "pct_test": round(pct_test, 1),
            "test_start": test_start_ts.date(),
            "test_end": test_end_ts.date(),
        }
    )

fold_stats_df = pd.DataFrame(fold_stats)
print(f"Total bars: {n}")
print(f"Embargo size: {embargo_size} bars (frac={EMBARGO_FRAC})")
print()
display(fold_stats_df.style.format({  # noqa: F821
    "pct_train": "{:.1f}%",
    "pct_test": "{:.1f}%",
}))

<a id='regime-backtest'></a>
---
# Part 3: Regime A/B Backtest Comparison

## Regime-Filtered Backtest Comparison

This section compares two versions of the same RSI signal:

- **Baseline**: RSI signal without regime filter — enters whenever `rsi_14 < 30` (oversold)
- **Regime-filtered**: Same RSI signal, but **blocks entries in Down-* regimes** (downtrend)

The hypothesis: entering long in a downtrend regime is a value trap. Blocking entries when the regime is bearish should reduce losing trades at the cost of missing some reversals.

**Exit condition**: `rsi_14 > 70` (overbought) — same for both strategies.

We then compare the two strategies on: **Sharpe ratio**, **Max Drawdown**, and **Number of Trades**.

In [ ]:
from ta_lab2.analysis.ic import load_regimes_for_asset

# Load regime labels
with engine.connect() as conn:
    regimes_df = load_regimes_for_asset(
        conn, ASSET_ID, TF, train_start, train_end
    )

if regimes_df.empty:
    print("WARNING: No regime data found. Regime filter will be skipped (all entries kept).")
    print("Run: python -m ta_lab2.scripts.regimes.refresh_cmc_regimes --all")
    HAS_REGIMES = False
else:
    HAS_REGIMES = True
    print(f"Loaded {len(regimes_df)} regime bars")
    trend_counts = regimes_df["trend_state"].value_counts()
    print(f"\nTrend state distribution:")
    for label, count in trend_counts.items():
        pct = 100 * count / len(regimes_df)
        print(f"  {label}: {count} bars ({pct:.1f}%)")
    regimes_df.head()

<a id='rsi-signals'></a>
## RSI Signal Generation

We generate entry and exit signals directly from `rsi_14` in `cmc_features`:

- **Entry**: `rsi_14 < 30` (oversold — contrarian buy signal)
- **Exit**: `rsi_14 > 70` (overbought — exit long position)

This is a classic mean-reversion strategy: buy when RSI is oversold, sell when it reaches overbought territory.

In [ ]:
RSI_COL = "rsi_14"
RSI_OVERSOLD = 30
RSI_OVERBOUGHT = 70

if RSI_COL not in features_df.columns:
    raise RuntimeError(
        f"RSI column '{RSI_COL}' not found in cmc_features. "
        "Run feature refresh to populate TA features."
    )

rsi = features_df[RSI_COL]

# RSI-based entry/exit signals
entries_base = rsi < RSI_OVERSOLD   # buy signal: oversold
exits = rsi > RSI_OVERBOUGHT        # sell signal: overbought

print(f"RSI signal stats (unfiltered):")
print(f"  Total bars: {len(rsi)}")
print(f"  Entry signals (RSI < {RSI_OVERSOLD}): {entries_base.sum()} bars ({100*entries_base.sum()/len(entries_base):.1f}%)")
print(f"  Exit signals (RSI > {RSI_OVERBOUGHT}): {exits.sum()} bars ({100*exits.sum()/len(exits):.1f}%)")

<a id='regime-filter'></a>
## Regime-Filtered Entries

The regime filter blocks entry signals in **downtrend regimes** (trend_state == 'downtrend'). Entries are kept for uptrend and sideways regimes.

In [ ]:
if HAS_REGIMES:
    # Align regime trend_state to feature index
    regime_trend = regimes_df["trend_state"].reindex(rsi.index)

    # Forward-fill regime (regime is sticky — holds until next bar with data)
    regime_trend = regime_trend.ffill()

    # Block entries in downtrend regime
    is_downtrend = regime_trend.str.lower().str.contains("down", na=False)
    entries_regime = entries_base & ~is_downtrend

    print(f"Regime filter effect:")
    print(f"  Entries (baseline, no filter): {entries_base.sum()}")
    print(f"  Bars in downtrend regime: {is_downtrend.sum()} ({100*is_downtrend.sum()/len(is_downtrend):.1f}%)")
    print(f"  Entries (regime-filtered):     {entries_regime.sum()}")
    print(f"  Entries blocked by regime:     {entries_base.sum() - entries_regime.sum()}")
else:
    print("No regime data — using baseline entries for both strategies.")
    entries_regime = entries_base.copy()

<a id='run-backtests'></a>
## Run Both Backtests

We use `run_vbt_on_split()` from `ta_lab2.backtests.vbt_runner` to run the vectorbt backtest.

**Critical vectorbt requirement**: All DataFrame and Series indexes must be **tz-naive** (no timezone). We strip the UTC timezone at the vectorbt boundary.

In [ ]:
try:
    from ta_lab2.backtests.vbt_runner import run_vbt_on_split, CostModel, Split

    # vectorbt requires tz-naive DatetimeIndex
    # Strip timezone at the vectorbt boundary
    def _strip_tz(s):
        """Remove timezone from a Series or DataFrame with DatetimeIndex."""
        if hasattr(s.index, "tz") and s.index.tz is not None:
            s = s.copy()
            s.index = s.index.tz_localize(None)
        return s

    # Build tz-naive versions for vectorbt
    features_naive = _strip_tz(features_df)
    entries_base_naive = _strip_tz(entries_base)
    exits_naive = _strip_tz(exits)
    entries_regime_naive = _strip_tz(entries_regime)

    # Cost model: 5bps fee + 5bps slippage per trade
    cost = CostModel(fee_bps=5, slippage_bps=5)

    # Single full-period split
    split = Split("full", pd.Timestamp(START_DATE), pd.Timestamp(END_DATE))

    # Run baseline (unfiltered RSI strategy)
    result_base = run_vbt_on_split(
        features_naive, entries_base_naive, exits_naive, None, cost, split
    )

    # Run regime-filtered strategy
    result_regime = run_vbt_on_split(
        features_naive, entries_regime_naive, exits_naive, None, cost, split
    )

    _backtest_ok = True
    print("Both backtests completed successfully.")
    print(f"  Baseline  — Sharpe: {result_base.sharpe:.4f}, Trades: {result_base.trades}")
    print(f"  Regime-filtered — Sharpe: {result_regime.sharpe:.4f}, Trades: {result_regime.trades}")

except Exception as exc:
    print(f"vectorbt backtest failed: {exc}")
    print("Skipping backtest comparison — vectorbt may not be installed or feature data is missing.")
    _backtest_ok = False

<a id='comparison-table'></a>
## Comparison Table

Key metrics for both strategies:
- **Sharpe** — risk-adjusted return (higher is better)
- **MDD** — Max Drawdown (less negative is better)
- **CAGR** — Compound Annual Growth Rate
- **MAR** — CAGR / |MDD| ratio (Calmar ratio; higher is better)
- **Trades** — number of completed trades
- **Total Return** — absolute return over the full period

In [ ]:
if _backtest_ok:
    comparison = pd.DataFrame(
        [
            {
                "Strategy": "Baseline (no regime filter)",
                "Sharpe": round(result_base.sharpe, 4),
                "MDD": round(result_base.mdd, 4),
                "CAGR": round(result_base.cagr, 4),
                "MAR": round(result_base.mar, 4),
                "Trades": result_base.trades,
                "Total Return": round(result_base.total_return, 4),
            },
            {
                "Strategy": "Regime-filtered (block downtrend entries)",
                "Sharpe": round(result_regime.sharpe, 4),
                "MDD": round(result_regime.mdd, 4),
                "CAGR": round(result_regime.cagr, 4),
                "MAR": round(result_regime.mar, 4),
                "Trades": result_regime.trades,
                "Total Return": round(result_regime.total_return, 4),
            },
        ]
    ).set_index("Strategy")

    print("Regime A/B Backtest Comparison")
    print(f"Period: {START_DATE} to {END_DATE}, Asset ID: {ASSET_ID}, TF: {TF}")
    print(f"Signal: RSI oversold entry (RSI < {RSI_OVERSOLD}) / overbought exit (RSI > {RSI_OVERBOUGHT})")
    print()

    # Style: highlight better value in green
    def highlight_better(df):
        styles = pd.DataFrame("", index=df.index, columns=df.columns)
        for col in ["Sharpe", "CAGR", "MAR", "Total Return"]:
            if col in df.columns:
                best_idx = df[col].idxmax()
                styles.loc[best_idx, col] = "background-color: #c6efce; font-weight: bold"
        for col in ["MDD"]:
            if col in df.columns:
                # Less negative = better for MDD
                best_idx = df[col].idxmax()
                styles.loc[best_idx, col] = "background-color: #c6efce; font-weight: bold"
        return styles

    styled_comparison = comparison.style.apply(highlight_better, axis=None)
    display(styled_comparison)  # noqa: F821

    # Delta row
    delta_sharpe = result_regime.sharpe - result_base.sharpe
    delta_mdd = result_regime.mdd - result_base.mdd
    delta_trades = result_regime.trades - result_base.trades
    print(f"\nDelta (regime-filtered minus baseline):")
    print(f"  Sharpe: {delta_sharpe:+.4f}")
    print(f"  MDD:    {delta_mdd:+.4f} (positive = less drawdown)")
    print(f"  Trades: {delta_trades:+d}")
else:
    print("Backtest did not run — skipping comparison table.")
    print("Install vectorbt to enable: pip install vectorbt")

**Reading the A/B comparison:**

- If the **regime-filtered strategy has higher Sharpe and less negative MDD**, regime filtering is adding value by avoiding bad entries.
- If the **regime-filtered strategy has fewer trades**, it's correctly blocking entries during downtrends — this is the intended behavior.
- If the **baseline outperforms**, it may indicate that oversold RSI during downtrends still recovers (perhaps the market is mean-reverting even in downtrend regimes), or the downtrend regime is too broad.

**Important caveats:**
1. This is a single in-sample evaluation. Purged K-Fold (Part 2) is needed for unbiased out-of-sample comparison.
2. Transaction costs (5bps fee + 5bps slippage) are included. Higher costs penalize strategies with more trades.
3. The regime filter is a crude on/off switch. A continuous regime score (e.g., P(uptrend) from a probabilistic classifier) would be more nuanced.

<a id='regime-ic'></a>
## Regime-Conditional IC

A natural extension of the A/B backtest: compute IC *separately for each regime*. This reveals whether the feature's predictive power is concentrated in specific market conditions.

We use `compute_ic_by_regime()` which accepts a pre-built `regimes_df` (trend_state parsed from l2_label).

In [ ]:
from ta_lab2.analysis.ic import compute_ic_by_regime

if HAS_REGIMES:
    print(f"Computing regime-conditional IC for feature: {FEATURE_COL}")
    print(f"  Regime column: trend_state (Up/Down/Sideways)")
    print(f"  Horizons: {HORIZONS[:4]} (first 4 for speed)")
    print()

    regime_ic_df = compute_ic_by_regime(
        feature_series,
        close,
        regimes_df,
        train_start,
        train_end,
        horizons=HORIZONS[:4],  # use first 4 horizons for speed
        return_types=["arith"],
        rolling_window=ROLLING_WINDOW,
        tf_days_nominal=1,
        regime_col="trend_state",
        min_obs_per_regime=30,
    )

    print(f"Regime-conditional IC results: {len(regime_ic_df)} rows")
    regime_ic_df.head(15)
else:
    print("No regime data available — skipping regime-conditional IC.")
    regime_ic_df = pd.DataFrame()

In [ ]:
if HAS_REGIMES and not regime_ic_df.empty:
    # Pivot: regime x horizon IC table
    regime_pivot = regime_ic_df.pivot_table(
        index="regime_label",
        columns="horizon",
        values="ic",
        aggfunc="mean",
    ).round(4)
    regime_pivot.columns = [f"h={c}" for c in regime_pivot.columns]

    # Add n_obs column (average across horizons)
    regime_pivot["mean_n_obs"] = regime_ic_df.groupby("regime_label")["n_obs"].mean().round(0).astype(int)

    print(f"Regime-Conditional IC Table — Feature: {FEATURE_COL} (arith returns)")
    print(f"Each row is a regime label (split from l2_label 'trend-vol' -> trend_state)")
    print()

    ic_cols_regime = [c for c in regime_pivot.columns if c.startswith("h=")]
    styled_regime = (
        regime_pivot.style
        .background_gradient(subset=ic_cols_regime, cmap="RdYlGn", vmin=-0.10, vmax=0.10)
        .format("{:.4f}", subset=ic_cols_regime)
        .format("{:,.0f}", subset=["mean_n_obs"])
    )
    display(styled_regime)  # noqa: F821

    print("\nInterpretation:")
    print("  Uptrend rows: Is the feature predictive during bull markets?")
    print("  Downtrend rows: Is the signal reversed in bear markets (mean-reversion vs momentum)?")
    print("  Sideways rows: Does the signal work in choppy/ranging conditions?")
else:
    print("Skipping regime IC table — no regime data available.")

**Interpretation of regime-conditional IC:**

If `rsi_14` shows:
- **Negative IC in uptrend**: High RSI → pullback (mean reversion works during uptrends)
- **Positive IC in downtrend**: High RSI → continuation down (momentum works, RSI fails to signal reversals)
- **Near-zero IC in sideways**: No predictive power in ranging markets

This regime decomposition is far more informative than a single IC value. A feature with IC = 0.03 overall might have IC = -0.08 in uptrends and IC = +0.06 in downtrends — a huge difference that the aggregate masks.

**Application**: If a feature has strong positive IC only in one regime, consider using it only when that regime is detected — effectively the same logic as the A/B backtest regime filter.

<a id='summary'></a>
---
## Summary and Next Steps

### What We Covered

**Part 1 — IC Evaluation:**

1. **Information Coefficient (IC)** — Spearman rank correlation between feature and forward returns. Boundary masking prevents future-information leakage (bars near `train_end` have their forward returns nulled).

2. **IC Decay** — Evaluated `rsi_14` across 7 forward horizons [1,2,3,5,10,20,60] bars. Significance coloring (royalblue = p<0.05) shows which horizons have detectable predictive signal.

3. **Rolling IC + IC-IR** — 63-bar rolling IC shows whether the feature's predictive power is stable over time. IC-IR > 0.5 indicates consistent (not regime-concentrated) signal.

4. **Batch IC** — Evaluated multiple features simultaneously with a pivot table showing the IC decay profile for each feature.

**Part 2 — Purged K-Fold:**

5. **Leakage-free splits** — `PurgedKFoldSplitter` with `t1_series` removes overlapping-label observations (purge) and serial autocorrelation buffer (embargo). Each fold has clearly separated train/test/purged/embargo zones.

6. **Fold visualization** — Matplotlib horizontal bar chart showing exactly which bars are in each category for each fold.

**Part 3 — Regime A/B Backtest:**

7. **RSI mean-reversion strategy** — Entry: RSI < 30, Exit: RSI > 70. Tested with and without downtrend regime filter.

8. **Regime-conditional IC** — IC computed separately per regime label, revealing whether the feature's predictive power is regime-dependent.

---

### Next Steps

**Notebook 03 (Backtest)** — Full backtest pipeline using Purged K-Fold for out-of-sample evaluation:
- Run the same RSI strategy across all 5 purged K-folds
- Aggregate fold-level Sharpe ratios to get an unbiased performance estimate
- PSR (Probabilistic Sharpe Ratio) to test whether observed Sharpe > 0 is statistically significant
- Equity curve across folds

**IC to feature promotion** — If a feature shows:
- IC-IR > 0.5 at h=5
- p-value < 0.05 after Benjamini-Hochberg correction

...promote to `dim_feature_registry` and add to `cmc_feature_experiments`:
```bash
python -m ta_lab2.scripts.experiments.promote_feature --feature rsi_14 --min-pass-rate 0.5
```

**Full IC evaluation (CLI):**
```bash
# Save IC results to cmc_ic_results table
python -m ta_lab2.scripts.ic.run_ic_eval \
    --asset-id 1 \
    --tf 1D \
    --train-start 2021-01-01 \
    --train-end 2024-12-31 \
    --feature rsi_14 \
    --all-features
```

**Streamlit dashboard** — Visualize IC results interactively:
```bash
streamlit run src/ta_lab2/dashboard/app.py
# Navigate to Research Explorer for IC decay charts and regime timeline
```